In [ ]:
!pip install python-docx

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 4.5 MB/s eta 0:00:00


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
from docx import Document
import os
import re
import glob
from docx.table import Table
from docx.text.paragraph import Paragraph

# iterates through paragraphs and tables in order
def iter_block_items(doc):
    body = doc.element.body
    for child in body.iterchildren():
        if child.tag.endswith('}p'):
            yield Paragraph(child, doc)
        elif child.tag.endswith('}tbl'):
            yield Table(child, doc)

def parse_roar_document(file_path, label):
    doc = Document(file_path)

    row_data = {
        "filename": os.path.basename(file_path),
        "department": None,
        "plo": "",
        "methods": "",
        "results_conclusions": "",
        "improvement_plan": "",
        "label": label
    }

    current_section = None

    def handle_text(text):
        nonlocal current_section
        text = (text or "").strip()
        if not text:
            return

        if "Department/Academic Program:" in text:
            row_data["department"] = text.split(":",1)[-1].strip()
            current_section = None
            return

        t = text.lower()

        if "specific program learning outcome" in t:
            current_section = "plo"
            return

        if t.startswith("methods"):
            current_section = "methods"
            return

        if "results and conclusions" in t:
            current_section = "results_conclusions"
            return

        if "improvement plan" in t:
            current_section = "improvement_plan"
            return

        if current_section:
            row_data[current_section] += text + " "

    for block in iter_block_items(doc):

        if isinstance(block, Paragraph):
            handle_text(block.text)

        elif isinstance(block, Table):
            for row in block.rows:
                for cell in row.cells:
                    for p in cell.paragraphs:
                        handle_text(p.text)

    for k in ("plo","methods","results_conclusions","improvement_plan"):
        row_data[k] = row_data[k].strip()

    row_data["plo"] = re.sub(
    r'^\s*\(?\b(?:PLO|SLO)\s*\d+[^\w]*',
    '',
    row_data["plo"]
).strip()

    row_data["methods"] = row_data["methods"].replace(
        "What kind of direct assessment did you use? (i.e. describe how you selected the sample of work that was assessed, the rubric used, etc. Best practices and SACSCOC requires the use of direct assessment.) If you also used indirect assessment, please describe that as well.",
        ""
    ).strip()

    row_data["results_conclusions"] = row_data["results_conclusions"].replace(
        "What were the results of your evaluation?",
        ""
    ).strip()

    row_data["improvement_plan"] = row_data["improvement_plan"].replace(
        "Based on your results, what changes (if any) will you make? What is your timeline to make these changes? What is your timeline for assessing the impact of these changes?",
        ""
    ).strip()

    return row_data

# function to get real docx files
def list_docx(folder):
    files = glob.glob(os.path.join(folder,"*.docx"))
    return [f for f in files if not os.path.basename(f).startswith("~$")]


# folder paths
good_folder = "/content/drive/MyDrive/Colab Notebooks/GOOD ROARS"
bad_folder  = "/content/drive/MyDrive/Colab Notebooks/BAD ROARS"

good_files = list_docx(good_folder)
bad_files = list_docx(bad_folder)

# parse documents
data_list = ([parse_roar_document(f,1) for f in good_files] +
             [parse_roar_document(f,0) for f in bad_files])

df = pd.DataFrame(data_list)

print("good:",len(good_files))
print("bad:",len(bad_files))

df.head(10)
df.to_csv("/content/drive/MyDrive/labeled_dataset.csv", index=False)

Good files: 26
Bad files: 13
label
1    26
0    13
Name: count, dtype: int64


In [ ]:
from google.colab import files

files.download('/content/drive/MyDrive/labeled_dataset.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>